# Chemistry Production Problem with AMPL in Python
## AMPL Modeling for Problem 1

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Chemistry Production Plan

A chemical company manufactures three products: A, B, and C using two processes.

- **Process 1** (1 hour): costs $40, produces 3 units of A, 1 of B, 1 of C
- **Process 2** (1 hour): costs $10, produces 1 unit of A, 1 of B

Daily demand requirements:
- At least 40 units of A
- At least 15 units of B
- At least 5 units of C

Minimize the daily production cost.

In [4]:
@timed
def solve_chemistry(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        var x1 >= 0;
        var x2 >= 0;

        minimize TotalCost:
            40 * x1 + 10 * x2;

        subject to DemandA:
            3 * x1 + x2 >= 40;

        subject to DemandB:
            x1 + x2 >= 15;

        subject to DemandC:
            x1 >= 5;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "x1": round(ampl.get_value("x1"), 4),
        "x2": round(ampl.get_value("x2"), 4),
        "cost": round(ampl.get_value("TotalCost"), 4),
    }
    return solution

In [5]:
result, elapsed = solve_chemistry()

print("=== CHEMISTRY PRODUCTION RESULTS ===")
print(f"Process 1 hours (x1) -> {result['x1']}")
print(f"Process 2 hours (x2) -> {result['x2']}")
print(f"Minimum daily cost   -> ${result['cost']}")
print(f"Time                 -> {elapsed:.8f} seconds")
print()
print("Production verification:")
print(f"  Product A: {3*result['x1'] + result['x2']:.1f} units (need >= 40)")
print(f"  Product B: {result['x1'] + result['x2']:.1f} units (need >= 15)")
print(f"  Product C: {result['x1']:.1f} units (need >= 5)")

HiGHS 1.11.0=== CHEMISTRY PRODUCTION RESULTS ===
Process 1 hours (x1) -> 5
Process 2 hours (x2) -> 25
Minimum daily cost   -> $450
Time                 -> 0.01301436 seconds

Production verification:
  Product A: 40.0 units (need >= 40)
  Product B: 30.0 units (need >= 15)
  Product C: 5.0 units (need >= 5)


## Sensitivity Analysis

In [6]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    var x1 >= 0;
    var x2 >= 0;

    minimize TotalCost:
        40 * x1 + 10 * x2;

    subject to DemandA:
        3 * x1 + x2 >= 40;

    subject to DemandB:
        x1 + x2 >= 15;

    subject to DemandC:
        x1 >= 5;
    '''
)
ampl.solve()

print("=== SHADOW PRICES (Dual Values) ===")
print(f"DemandA shadow price: {ampl.get_constraint('DemandA').dual():.4f}")
print(f"DemandB shadow price: {ampl.get_constraint('DemandB').dual():.4f}")
print(f"DemandC shadow price: {ampl.get_constraint('DemandC').dual():.4f}")
print()
print("=== CONSTRAINT SLACK ===")
print(f"DemandA slack: {ampl.get_constraint('DemandA').slack():.4f}")
print(f"DemandB slack: {ampl.get_constraint('DemandB').slack():.4f}")
print(f"DemandC slack: {ampl.get_constraint('DemandC').slack():.4f}")
print()
print("=== REDUCED COSTS ===")
print(f"x1 reduced cost: {ampl.get_variable('x1').rc():.4f}")
print(f"x2 reduced cost: {ampl.get_variable('x2').rc():.4f}")

HiGHS 1.11.0=== SHADOW PRICES (Dual Values) ===
DemandA shadow price: 10.0000
DemandB shadow price: 0.0000
DemandC shadow price: 10.0000

=== CONSTRAINT SLACK ===
DemandA slack: 0.0000
DemandB slack: 15.0000
DemandC slack: 0.0000

=== REDUCED COSTS ===
x1 reduced cost: 0.0000
x2 reduced cost: 0.0000


## Interpretation

The shadow prices indicate the marginal cost of increasing each demand requirement by one unit:

- **DemandA**: The shadow price shows how much the total cost would increase if the company needed one more unit of Product A per day.
- **DemandB**: Similarly for Product B.
- **DemandC**: Product C is only produced by Process 1, which is the more expensive process, so this constraint directly impacts the cost.

A zero slack means the constraint is binding (active at the optimum). A positive slack means the constraint is not tight — the production exceeds the minimum demand.